# Exercise 01 - Introduction to Finite Element Analysis
Todays task is to implement a  simple FE-Code for a 1D-elasticity problem. Therefore, we look at the following picture:

<img src="fig/exercise1-1.jpeg">

The displayed rod is made of steel ( $E=200 GPa$ ). Its diameter is $d=9 mm$, the length is $L=750 mm$. The loading is given by a singular force on the right hand side with $P=100kN$.

### Task 1.1
Transofrm the given properties to a consistent SI-unit system. What kind of time-scale did you use?

In [1]:
import numpy as np

E = 200000 # Youngs modulus in MPa
P = 100000 # applied load in N
d = 9 # diameter in mm
A = np.pi * (d/2)**2    # cross-sectional
L = 750 # length in mm

### Task 2.2
Define a discretization for the problem at hand. How many elements do you want to use compute displacements, stresses and strains?
With this, define your element geometry (length), and the number of nodes in your FE-model.
Additionally, define your boundary conditions.

In [2]:
n = [] # node coordinates
n_el = 20 # number of elements

for i in range(n_el + 1):
    n.append(i * L / n_el)

# global load vector
f = np.zeros(len(n))
f[-1] = P



### Task 2.3
Write two classes: A node and a element class. Remember to introduce some connectivity between nodes and elements. 


In [7]:
import numpy as np

class Node:
    """A 1D FE node with a single translational DOF (u).
    ----------------
    args:
        id: int
            node ID
        x: float  
            coordinate in [mm]
        elements: list
            list of the connected elements
    """
    def __init__(self, id, x, elements):
        self.id = id
        self.x = x
        self.elements = elements
    
    def add_element(self, element):
        """Register that this node is connected to an element."""
        if element not in self.elements:
            self.elements.append(element)

    


class Element:
    """Two-node 1D bar (rod) element with axial deformation."""

    def __init__(self, id, n1, n2, E, A):
        if n1.id == n2.id:
            raise ValueError("An element must connect two different nodes.")
        self.id = id
        self.nodes = (n1, n2)
        self.E = E  # Young's modulus [MPa]
        self.A = A  # cross-sectional area [mm^2]

        # Two-way connectivity: element knows its nodes, nodes know their elements
        n1.add_element(self)
        n2.add_element(self)
        self.length = abs(n2.x - n1.x)

    def element_stiffness_matrix(self):
        """Element stiffness matrix (2x2) k_e."""
        L = self.length
        if L <= 0.0:
            raise ValueError("Element length must be positive.")
        k = (self.E * self.A / L) * np.array([[1.0, -1.0], [-1.0, 1.0]])
        return k
    
    def element_load_vector(self, P1):
        """Element load vector (2x1) f_e."""
        f = np.array([0.0, 0.0])
        f[1] = P




### Task 2.4
Write a model class, that creates a discretization of the problem as done in Task 2.2 and then sets up the elements and nodes.
Remember to implement an overview over the elements and nodes that you create.

### Task 2.5
Assemble the global stiffness matrix.

### Task 2.6
Incorporate boundary conditions into your model to avoid rigid body motions.

In [13]:
class Model:
    """A 1D FE model consisting of nodes and elements."""

    def __init__(self):
        self.nodes = []
        self.elements = []

    def create_rod_mesh(self, n_el, L, E, A):
        """Create a 1D mesh of rod elements.
        ----------------
        args:
            n_el: int
                number of elements
            L: float
                total length of the rod in [mm]
            E: float
                Young's modulus in [MPa]
            A: float
                cross-sectional area in [mm^2]
            F: np.array
                global force vector [N]
        """
        # Create nodes
        for i in range(n_el + 1):
            x = i * L / n_el
            node = Node(id=i, x=x, elements=[])
            self.nodes.append(node)

        # Create elements
        for i in range(n_el):
            n1 = self.nodes[i]
            n2 = self.nodes[i + 1]
            element = Element(id=i, n1=n1, n2=n2, E=E, A=A)
            self.elements.append(element)
    
    def assemble_global_stiffness(self):
        """Assemble the global stiffness matrix K."""
        n_nodes = len(self.nodes)
        K = np.zeros((n_nodes, n_nodes))

        for element in self.elements:
            k_e = element.element_stiffness_matrix()
            n1_id = element.nodes[0].id
            n2_id = element.nodes[1].id

            # Add element stiffness to global stiffness matrix
            K[n1_id, n1_id] += k_e[0, 0]
            K[n1_id, n2_id] += k_e[0, 1]
            K[n2_id, n1_id] += k_e[1, 0]
            K[n2_id, n2_id] += k_e[1, 1]
        self.K = K
    

    def apply_boundary_conditions(self, fixed_nodes, P):
        """Apply boundary conditions by modifying the stiffness matrix and load vector.
        ----------------
        args:
            fixed_nodes: list of int
                list of node IDs that are fixed (u=0)
        """

        self.F = np.zeros(len(self.nodes))
        self.F[-1] = P

        self.K_regularized = self.K.copy()
        for node_id in fixed_nodes:
            self.K_regularized[node_id, :] = 0.0
            self.K_regularized[:, node_id] = 0.0
            self.K_regularized[node_id, node_id] = 1.0
            self.F[node_id] = 0.0



### Task 2.7
Solve your model by inverting the global stiffness matrix. Additionally, compute the strains and stresses in the elements. Last, calculate the nodal force vectors and compute the reaction force at the support. 

In [19]:
from numpy.linalg import inv

fe_model = Model()
fe_model.create_rod_mesh(n_el=10, L=L, E=E, A=A)
fe_model.assemble_global_stiffness()
fe_model.apply_boundary_conditions(fixed_nodes=[0], P=P) 

displacements = np.linalg.inv(fe_model.K_regularized) @ fe_model.F
print("Nodal displacements (mm):", displacements)


# Strains and stresses in each element
for element in fe_model.elements:
    n1_id = element.nodes[0].id
    n2_id = element.nodes[1].id
    u1 = displacements[n1_id]
    u2 = displacements[n2_id]
    strain = (u2 - u1) / element.length
    stress = element.E * strain
    print(f"Element {element.id}: Strain = {strain:.6e}, Stress = {stress:.2f} MPa")

# Nodal forces and reaction force for the fixed nodes
nodal_forces = fe_model.K @ displacements
print("Nodal forces (N):", nodal_forces)
print("Reaction force at fixed node (N):", nodal_forces[0])

Nodal displacements (mm): [0.         0.58946275 1.1789255  1.76838826 2.35785101 2.94731376
 3.53677651 4.12623927 4.71570202 5.30516477 5.89462752]
Element 0: Strain = 7.859503e-03, Stress = 1571.90 MPa
Element 1: Strain = 7.859503e-03, Stress = 1571.90 MPa
Element 2: Strain = 7.859503e-03, Stress = 1571.90 MPa
Element 3: Strain = 7.859503e-03, Stress = 1571.90 MPa
Element 4: Strain = 7.859503e-03, Stress = 1571.90 MPa
Element 5: Strain = 7.859503e-03, Stress = 1571.90 MPa
Element 6: Strain = 7.859503e-03, Stress = 1571.90 MPa
Element 7: Strain = 7.859503e-03, Stress = 1571.90 MPa
Element 8: Strain = 7.859503e-03, Stress = 1571.90 MPa
Element 9: Strain = 7.859503e-03, Stress = 1571.90 MPa
Nodal forces (N): [-1.00000000e+05  2.91038305e-11  0.00000000e+00  0.00000000e+00
  0.00000000e+00  0.00000000e+00  0.00000000e+00  0.00000000e+00
  0.00000000e+00 -2.32830644e-10  1.00000000e+05]
Reaction force at fixed node (N): -99999.99999999984
